# YOLO11s 베이스라인 학습

**전제:** 팀원 노트북(`pill_detection_dataset.ipynb`)을 실행해 `data/processed` 아래에 
`images/{train,val,test}`, `labels/{train,val,test}`, `data.yaml` 이 생성돼 있어야 합니다.
이 노트북(`notebooks/` 안에 위치)은 그 `data.yaml`만 받아 YOLO11s를 학습합니다.

베이스라인은 팀 합의대로 **기본값 위주**로 돌립니다.


## 1. 설치 · 환경 · 재현성


In [1]:
# Colab이면 실행 (로컬은 최초 1회만)
!pip -q install ultralytics

import os, random, numpy as np, torch
from pathlib import Path
from ultralytics import YOLO

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| device:", DEVICE)

torch: 2.13.0 | CUDA: False | device: cpu


## 2. data.yaml 경로 이식성 처리

업로드된 `data.yaml`의 `path`는 팀원 PC의 Windows 절대경로라 그대로 쓰면 실패합니다.
현재 런타임의 실제 위치로 `path`만 덮어써서 `data_runtime.yaml`로 다시 저장합니다.


In [2]:
import yaml

# YOLO 데이터셋 폴더 (images/train, labels/train ... 가 들어있는 곳)
DATASET_DIR = Path("../data/processed")   # notebooks/ 기준 상대경로. Colab이면 절대경로로 수정.

src_yaml = DATASET_DIR / "data.yaml"
cfg = yaml.safe_load(open(src_yaml, encoding="utf-8"))
cfg["path"] = str(DATASET_DIR.resolve())
RUNTIME_YAML = DATASET_DIR / "data_runtime.yaml"
yaml.safe_dump(cfg, open(RUNTIME_YAML, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)

assert (DATASET_DIR / "images" / "train").is_dir(), \
    "images/train 이 없습니다. 팀원 노트북(셀 6)을 먼저 실행하세요."
print("클래스 수 :", cfg["nc"])
print("학습 yaml :", RUNTIME_YAML.resolve())

클래스 수 : 56
학습 yaml : /Users/wonyounglee/Documents/coding/codeit/projects/pill-object-detection/data/processed/data_runtime.yaml


## 3. 베이스라인 학습 (YOLO11s, 기본값)

`yolo11s.pt`는 COCO 사전학습 가중치로 첫 실행 시 자동 다운로드됩니다.


In [1]:
# W&B 설치 및 로그인 
!pip -q install wandb
import wandb
from wandb.integration.ultralytics import add_wandb_callback
wandb.login()   # 최초 1회, wandb.ai/authorize 의 API 키 붙여넣기

NameError: name 'ClassificationTrainer' is not defined

In [ ]:
# 팀원과 같은 W&B 프로젝트에 남기기
wandb.init(project="pill-object-detection", name="yolo11s-baseline",
           job_type="train", tags=["baseline", "yolo11s"])
           # entity="diokim17-org"  # 팀 워크스페이스에 모으려면 팀에 초대받은 뒤 주석 해제

model = YOLO("yolo11s.pt")

results = model.train(
    data=str(RUNTIME_YAML),
    epochs=100,          # 베이스라인 기본값. 빠른 점검은 10~20.
    imgsz=640,
    batch=16,            # 메모리 부족하면 8 또는 -1(자동)
    seed=SEED,
    deterministic=True,
    device=DEVICE,
    project="../outputs/yolo",
    name="yolo11s_baseline_noaug", #_noag 는 증강없이 라는 뜻
    exist_ok=True,
    # ── 기본 augmentation 모두 끄기 ──
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.0,
    mosaic=0.0, mixup=0.0, copy_paste=0.0, erasing=0.0
)
print("결과 폴더:", results.save_dir)

Ultralytics 8.4.116 🚀 Python-3.11.15 torch-2.13.0 CPU (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/processed/data_runtime.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_baseline, nbs=64, nms=False, opset

## 4. 검증 지표 (mAP)


In [ ]:
m = model.val(data=str(RUNTIME_YAML), split="val", device=DEVICE)
print(f"최종 Best Validation mAP@0.5:0.95: {m.box.map:.4f}") # ← 베이스라인 기준선
print(f"          mAP@0.5       : {m.box.map50:.4f}")
print(f"          mAP@0.75      : {m.box.map75:.4f}")
wandb.finish()

## 5. 예측 시각화 (검증 이미지 몇 장, max_det=4)


In [5]:
val_imgs = sorted((DATASET_DIR / "images" / "val").glob("*.png"))[:6]
pred = model.predict(
    val_imgs, imgsz=640, conf=0.25, max_det=4, device=DEVICE,
    save=True, project="../outputs/yolo", name="pred_val", exist_ok=True,
)
print("시각화 저장 위치:", pred[0].save_dir)


0: 640x512 1 일양하이트린정 2mg, 1 뉴로메드정(옥시라세탐), 1 오마코연질캡슐(오메가-3-산에틸에스테르90), 1 아토젯정 10/40mg, 111.3ms
1: 640x512 1 일양하이트린정 2mg, 1 뉴로메드정(옥시라세탐), 1 아토르바정 10mg, 111.3ms
2: 640x512 1 일양하이트린정 2mg, 1 뉴로메드정(옥시라세탐), 1 아토르바정 10mg, 111.3ms
3: 640x512 1 일양하이트린정 2mg, 1 뉴로메드정(옥시라세탐), 1 아토르바정 10mg, 111.3ms
4: 640x512 1 일양하이트린정 2mg, 1 에빅사정(메만틴염산염)(비매품), 1 플라빅스정 75mg, 111.3ms
5: 640x512 1 일양하이트린정 2mg, 1 종근당글리아티린연질캡슐(콜린알포세레이트) , 1 플라빅스정 75mg, 111.3ms
Speed: 1.4ms preprocess, 111.3ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 512)
Results saved to /Users/wonyounglee/Documents/coding/codeit/projects/pill-object-detection/notebooks/runs/outputs/yolo/pred_val
시각화 저장 위치: /Users/wonyounglee/Documents/coding/codeit/projects/pill-object-detection/notebooks/runs/outputs/yolo/pred_val


## 6. Kaggle 제출 파일 생성

제출 형식: `annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score`
- 한 행 = 객체 1개, `image_id` = 파일명 숫자(예: `1.png` → 1)
- `annotation_id` = 행마다 고유한 값, `bbox_*` = 좌상단 x,y + 너비,높이(픽셀), `score` = 신뢰도

> ⚠️ **`category_id` 매핑을 반드시 확인하세요.** YOLO는 0~55를 예측합니다. 팀원 Dataset이
> `label_offset=1`을 썼으므로 **`category_id = YOLO클래스 + 1`(=1~56)** 일 가능성이 높지만,
> 대회의 `sample_submission` 또는 클래스표로 최종 확인하세요. 클래스표(약 이름↔category_id)가
> 있으면 `data.yaml`의 `names[i]`로 **이름 기준 매핑**하는 게 가장 안전합니다.


In [8]:
import csv

KAGGLE_TEST_DIR = Path("../data/dataset/cleaning_data/test_images")   # ← 842장 테스트 이미지 경로로 수정

def to_category_id(yolo_cls: int) -> int:
    # ⚠️ 대회 체계에 맞게 확인/수정 (기본 가정: label_offset=1 → +1)
    return yolo_cls + 1

preds = model.predict(
    sorted(KAGGLE_TEST_DIR.glob("*.png"), key=lambda p: int(p.stem)),
    imgsz=640, conf=0.25, max_det=4, device=DEVICE, stream=True,
)

rows, ann_id = [], 1
for r in preds:
    image_id = int(Path(r.path).stem)                 # 파일명 숫자
    b = r.boxes
    for xyxy, conf, cls in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()):
        x1, y1, x2, y2 = xyxy
        rows.append([
            ann_id, image_id, to_category_id(int(cls)),
            int(round(x1)), int(round(y1)),               # bbox_x, bbox_y
            int(round(x2 - x1)), int(round(y2 - y1)),     # bbox_w, bbox_h
            round(float(conf), 4),                        # score
        ])
        ann_id += 1

out = Path("../outputs/submissions/submission.csv")
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["annotation_id","image_id","category_id","bbox_x","bbox_y","bbox_w","bbox_h","score"])
    w.writerows(rows)
print("작성 완료:", out.resolve(), "| 행 수:", len(rows))

: 

In [ ]:
""" 
커널 크래시 발생시 사용 --> 맨윗줄과 맨아랫줄의 따옴표 제거

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # macOS OpenMP 충돌로 인한 커널 크래시 예방
import csv
from pathlib import Path
from ultralytics import YOLO

# 학습된 가중치 로드 (재학습 X). 경로는 셀3 출력의 "결과 폴더" + /weights/best.pt
model = YOLO("runs/outputs/yolo/yolo11s_baseline/weights/best.pt")

KAGGLE_TEST_DIR = Path("../data/dataset/cleaning_data/test_images")
test_files = sorted(KAGGLE_TEST_DIR.glob("*.png"), key=lambda p: int(p.stem))
assert test_files, f"이미지 없음: {KAGGLE_TEST_DIR.resolve()}"
print("테스트 이미지:", len(test_files))

def to_category_id(yolo_cls: int) -> int:
    return yolo_cls + 1

rows, ann_id = [], 1
for r in model.predict(test_files, imgsz=640, conf=0.25, max_det=4,
                       device=DEVICE, stream=True, verbose=False):   # verbose=False로 부하↓
    image_id = int(Path(r.path).stem)
    b = r.boxes
    for xyxy, conf, cls in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()):
        x1, y1, x2, y2 = xyxy
        rows.append([ann_id, image_id, to_category_id(int(cls)),
                     int(round(x1)), int(round(y1)),
                     int(round(x2 - x1)), int(round(y2 - y1)), round(float(conf), 4)])
        ann_id += 1

out = Path("../outputs/submissions/submission.csv"); out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["annotation_id","image_id","category_id","bbox_x","bbox_y","bbox_w","bbox_h","score"])
    w.writerows(rows)
print("작성:", out.resolve(), "| 행:", len(rows))

"""

테스트 이미지: 842


: 